In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalTransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        
        # 1. Attention Components
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # 2. Normalization (Pre-LN style)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        # 3. The MLP (Section 5)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.GELU(), # Modern transformers use GELU instead of ReLU
            nn.Linear(d_ff, d_model, bias=False)
        )

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        
        # --- PART 1: Causal Multi-Head Attention ---
        
        # 1. Apply LayerNorm (Pre-LN)
        x_norm = self.ln1(x)
        
        # 2. Project to Q, K, V
        Q = self.W_q(x_norm)
        K = self.W_k(x_norm)
        V = self.W_v(x_norm)
        
        # 3. Reshape for Multi-Head Attention
        # YOUR TASK: Reshape Q, K, V from (batch, seq, d_model) to (batch, num_heads, seq, d_k)
        # Hint: d_k = d_model // num_heads. Use .view() or .reshape()
        batch_size, seq_len, d_model = x.shape
        d_k = d_model // num_heads
        
        Q = Q.view(...) # Fill this in
        K = K.view(...) # Fill this in
        V = V.view(...) # Fill this in
        
        # 4. Scaled Dot-Product Attention with Causal Mask
        # PyTorch's magic function. It handles the scaling (1/sqrt(d_k)) AND the causal mask!
        # is_causal=True automatically builds the lower-triangular mask you wrote in NumPy.
        attn_output = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
        
        # 5. Concatenate Heads
        # YOUR TASK: Reshape attn_output back to (batch, seq, d_model)
        attn_output = attn_output.view(...) # Fill this in
        
        # 6. Residual Connection
        x = x + attn_output
        
        # --- PART 2: The MLP ---
        
        # 1. Apply LayerNorm
        x_norm = self.ln2(x)
        
        # 2. Pass through MLP
        mlp_output = self.mlp(x_norm)
        
        # 3. Residual Connection
        x = x + mlp_output
        
        return x

# --- Let's test it! ---
d_model = 64
num_heads = 4
d_ff = 256
batch_size = 2
seq_len = 10

# Create the block and move to your M5 GPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
block = CausalTransformerBlock(d_model, num_heads, d_ff).to(device)

# Create a dummy input (random noise)
x = torch.randn(batch_size, seq_len, d_model, device=device)

# Run the forward pass
output = block(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")